In [1]:
######################################################################
# 🚀 HMM $1M PAPER - FORCE REBALANCE TODAY!
######################################################################

import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from pandas.tseries.offsets import BDay
from datetime import datetime, timedelta
import os

from scripts.models.rolling_hmm import full_rolling_hmm_portfolio

warnings.filterwarnings('ignore')

# =============================================================================
# YOUR FUNCTIONS (unchanged)
# =============================================================================

def compute_portfolio_returns(weights_df, asset_ret_df):
    """weights ffill until next trade date (e.g. 31 Dec → 29 Jan)"""
    asset_ret_df = asset_ret_df.copy()
    asset_ret_df.columns = [col.replace('_ret', '').strip() for col in asset_ret_df.columns]
    
    common_assets = sorted(set(weights_df.columns) & set(asset_ret_df.columns))
    w = weights_df[common_assets].copy()
    r = asset_ret_df[common_assets].copy()
    
    # KEY: Reindex to ALL return dates + ffill (holds until next weight row)
    w_aligned = w.reindex(r.index, method='ffill')  # or .ffill() after reindex
    port_ret = (w_aligned * r).sum(axis=1)
    return pd.Series(port_ret, name="portfolio_return")


def summarize_performance(port_ret, rf_annual=0.0, periods_per_year=252):
    """COMPLETE: Sharpe + Annual Return + Vol + Max DD"""
    mean_ret = port_ret.mean()
    vol = port_ret.std()
    
    if pd.isna(mean_ret) or vol == 0:
        return {
            "sharpe": 0, 
            "ann_return": 0, 
            "ann_vol": 0,      # ✅ ADDED
            "max_drawdown": 0
        }
    
    # Annualized metrics
    ann_ret = (1 + mean_ret)**periods_per_year - 1
    ann_vol = vol * np.sqrt(periods_per_year)      # ✅ ANNUAL VOLATILITY
    rf_periodic = (1 + rf_annual)**(1/periods_per_year) - 1
    excess = port_ret - rf_periodic
    sharpe = excess.mean() / vol * np.sqrt(periods_per_year)
    
    # Max drawdown
    cum = (1 + port_ret).cumprod()
    dd = (cum / cum.cummax()) - 1
    max_dd = dd.min()
    
    return {
        "sharpe": sharpe,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,        # ✅ NEW!
        "max_drawdown": max_dd
    }


# =============================================================================
# FORCE REBALANCE TODAY
# =============================================================================

def force_rebalance(portfolio_value=1000000):
    print("🚨 FORCE $1M HMM REBALANCE")
    print("=" * 60)
    
    # 1. Run HMM
    fresh_data = pd.read_csv("../data/processed/selected_feature_matrix.csv", index_col=0, parse_dates=True)
    fresh_data_ret = fresh_data[[col for col in fresh_data.columns if col.endswith('_ret')]]
    signal = full_rolling_hmm_portfolio(fresh_data.tail(500), fresh_data_ret.tail(500)).iloc[-1]
    weights = signal.drop('regime')
    
    print("📊 HMM WEIGHTS:")
    print(weights.round(3))

    filepath = "../data/processed/actual_trades_log.csv"
    
    date = pd.Timestamp.now().normalize()
    weights_df = pd.DataFrame([weights])
    weights_df['regime'] = signal['regime']
    weights_df.index = [date]
    weights_df.index.name = 'date'
    
    # 🆕 Create dir + handle file logic
    if os.path.exists(filepath):
        existing_df = pd.read_csv(filepath, index_col=0, parse_dates=True)
        print(f"📖 Loaded {len(existing_df)} rows")
        weights_df = pd.concat([existing_df, weights_df])
    else:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        existing_df = pd.DataFrame()
        print("📖 New file")
    
    weights_df.to_csv(filepath)
    print(f"💾 Saved {len(weights_df)} rows")
    
    # 2. Calculate trades
    prev_weights = pd.Series(0.0, index=weights.index)
    if len(weights_df) > 1:
        prev_weights = weights_df.iloc[-2].drop('regime', errors='ignore')

    trades_df, trade_details = enhanced_transition_trades(prev_weights, weights, portfolio_value)
    print("🔍 DELTA CONFIRM:")
    for asset in trades_df.index:
        delta = trades_df.loc[asset, 'curr_w'] - trades_df.loc[asset, 'prev_w']
        print(f"{asset}: {delta:+.0%} (matches trade_size=${trades_df.loc[asset, 'trade_size']:,.0f})")

    # 3. LOG WEIGHTS (actual_trades_log.csv)
    #signal_df = pd.DataFrame([signal]).T
    #signal_df.columns = [pd.Timestamp.now().normalize()]
    #log = smart_trade_log(signal_df)
    
    # 4. LOG TRADES (executed_trades.csv)
    trade_details.to_csv("../data/processed/executed_trades.csv", index=False)
    print(f"\n✅ LOGGED:")
    print(f"   📈 actual_trades_log.csv → {len(weights)} HMM weights")
    print(f"   📋 executed_trades.csv → {len(trade_details)} orders")
    
    print(f"\n🎯 $1M DEPLOYED | Next rebalance: +21 business days")
    return trades_df, weights, trade_details

def log_daily_performance(portfolio_value=1000000):
    trades_log = pd.read_csv("../data/processed/actual_trades_log.csv", index_col=0, parse_dates=True)
    portfolio_assets = trades_log.drop('regime', axis=1, errors='ignore').columns.tolist()
    
    features = pd.read_csv("../data/processed/feature_matrix.csv", index_col=0, parse_dates=True)
    asset_ret = features.filter(like='_ret').tail(60)
    asset_ret.columns = [col.replace('_ret','').strip() for col in asset_ret.columns]
    
    common_assets = [asset for asset in portfolio_assets if asset in asset_ret.columns]
    if not common_assets:
        print("⚠️ No matching assets")
        return
    
    asset_ret = asset_ret[common_assets]
    weights_df = trades_log.drop('regime', axis=1, errors='ignore')
    port_ret = compute_portfolio_returns(weights_df, asset_ret)
    df = asset_ret.join(port_ret.rename('portfolio_return')).dropna()
    df = df.reset_index()
    cumulative_return = (1 + df['portfolio_return']).prod() 
    
    perf = summarize_performance(df['portfolio_return'])
    
    # ✅ FIXED: True days tracked = log length + 1
    filepath = "../data/processed/daily_performance_log.csv"
    try:
        existing_log = pd.read_csv(filepath, index_col=0, parse_dates=True)
        days_tracked = len(existing_log) + 1  # Day 1, 2, 3...
    except FileNotFoundError:
        days_tracked = 1
    
    # ✅ DAILY LOG ROW
    daily_log = pd.DataFrame([{
        'date': df['Date'].iloc[-1],
        'days_tracked': days_tracked,  # ✅ 1,2,3,4...
        'latest_return': df['portfolio_return'].iloc[-1],
        'cumulative_return': cumulative_return,
        'sharpe': perf['sharpe'],
        'ann_return': perf['ann_return'],
        'ann_vol': perf['ann_vol'],
        'max_drawdown': perf['max_drawdown'],
        'portfolio_value': portfolio_value * cumulative_return
    }])
    
    # ✅ APPEND
    try:
        existing_log = pd.read_csv(filepath, index_col=0, parse_dates=True)
        daily_log = pd.concat([existing_log, daily_log])
    except FileNotFoundError:
        pass
    
    daily_log.to_csv(filepath)
    
    # ✅ CLEAN PRINT (no dtype issues)
    latest = daily_log.iloc[-1]
    print(f"✅ DAY {days_tracked}:")
    print(f"   📊 Days: {int(latest['days_tracked'])}")
    print(f"   📈 Sharpe: {latest['sharpe']:.2f}")
    print(f"   💰 Value: ${latest['portfolio_value']:,.0f}")
    print(f"   📊 Latest: {latest['latest_return']:+.2%}")


def show_latest_performance():
    """Print latest stats from log"""
    try:
        perf_log = pd.read_csv("../data/processed/daily_performance_log.csv", index_col=0, parse_dates=True)
        latest = perf_log.iloc[-1]
        print(f"\n🎯 $1M PAPER TRADING:")
        print(f"   📊 Day {int(latest['days_tracked'])} | Value: ${latest['portfolio_value']:,.0f}")
        print(f"   📈 Sharpe: {latest['sharpe']:.2f} | Return: {latest['ann_return']:.1%}")
        print(f"   📉 Vol: {latest['ann_vol']:.1%} | Max DD: {latest['max_drawdown']:.1%}")
        print(f"   📊 Latest: {latest['latest_return']:+.2%}")
    except FileNotFoundError:
        print("📝 No log yet - first run!")


# =============================================================================
# TRADING FUNCTIONS (from before)
# =============================================================================

def get_trade_prices(ticker):
    try:
        data = yf.Ticker(ticker).history(period="5d")
        current = data['Close'].iloc[-1]
        high = data['High'].tail(3).max()
        low = data['Low'].tail(3).min()
        return {'current': round(float(current), 2), 'high_3d': round(float(high), 2)}
    except:
        return {"error": "API fail"}

def enhanced_transition_trades(prev_weights, curr_weights, portfolio_value=1000000):
    trades = transition_trades(prev_weights, curr_weights, portfolio_value)
    print("\n🎯 LIMIT + STOP ORDERS (IBKR Paper):")
    print("="*80)
    trade_details = [] # ✅ CREATE DataFrame
    
    for _, row in trades.iterrows():
        ticker = row.name
        prices = get_trade_prices(ticker)
        
        if 'error' not in prices and pd.notna(prices.get('current')):
            current = float(prices['current'])  # Ensure numeric
            high_3d = float(prices.get('high_3d', current))  # Fallback to current
            
            if row['trade_size'] > 0:  # BUY
                limit = round((high_3d + current) / 2, 2)
                stop = round(current * 0.85, 2)
            else:  # SELL/SHORT
                limit = round((current + high_3d) / 2, 2)
                stop = round(current * 1.15, 2)
        else:
            # Fallback on error/NaN
            current = limit = stop = 100.0
            print(f"⚠️ Price error → defaults used")
            
            print(f"{row['action']:>8} {ticker:>5} | ${row['trade_size']:>10,.0f} | {row['shares']:>6.0f} shares")
            print(f"   📊 Current: ${current:>6.2f} | Limit: ${limit:>6.2f} | Stop: ${stop:>6.2f}")
            print(f"   📋 ORDER: {row['shares']:>+6.0f} {ticker} @ LMT ${limit:.2f} | STP ${stop:.2f}")
            print()
            
            # ✅ ADD TO DataFrame
        trade_details.append({
                'ticker': ticker, 'action': row['action'],
                'shares': int(row['shares']),'price':row['price'], 'trade_size': row['trade_size'],
                'current': current, 'limit': limit, 'stop': stop
        })  
    
    # ✅ FIXED: RETURN trade_details DataFrame
    return trades, pd.DataFrame(trade_details)

def transition_trades(prev_weights, curr_weights, portfolio_value=1000000):
    """FIXED: Real prices from get_trade_prices() → CORRECT shares"""
    all_assets = sorted(set(prev_weights.index) | set(curr_weights.index))
    print(f"📊 Assets: {all_assets}")
    
    prev_w = prev_weights.reindex(all_assets).fillna(0)
    curr_w = curr_weights.reindex(all_assets).fillna(0)
    
    prev_value = portfolio_value * prev_w
    target_value = portfolio_value * curr_w
    
    # ✅ REAL PRICES from get_trade_prices()
    shares = {}
    prices = {}
    for asset in all_assets:
        price_data = get_trade_prices(asset)
        if 'error' not in price_data:
            current_price = price_data['current']
        else:
            current_price = 100  # Fallback
        prices[asset] = current_price
        prev_shares = (prev_value[asset] / current_price).round(0)
        target_shares = (target_value[asset] / current_price).round(0)
        trade_shares = target_shares - prev_shares
        
        shares[asset] = trade_shares
    
    trades = pd.DataFrame({
        'prev_w': prev_w.round(3), 
        'curr_w': curr_w.round(3),
        'prev_value': prev_value.round(0),
        'target_value': target_value.round(0),
        'trade_size': (target_value - prev_value).round(0),
        'shares': pd.Series(shares),
        'price': pd.Series(prices)
    }, index=all_assets)
    
    trades['action'] = trades['trade_size'].apply(
        lambda x: 'BUY' if x > 10 else 'SELL' if x < -10 else 'HOLD'
    )
    
    print(f"✅ {len(all_assets)} assets | Gross: ${trades['target_value'].abs().sum():,.0f}")
    return trades


def smart_trade_log(signal_df, filepath="../data/processed/actual_trades_log.csv"):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    if not os.path.exists(filepath):
        signal_df.index.name = 'date'
        signal_df.to_csv(filepath)
        print(f"✅ CREATED {os.path.basename(filepath)}")
        return pd.read_csv(filepath, index_col=0, parse_dates=True)
    else:
        log = pd.read_csv(filepath, index_col=0, parse_dates=True)
        new_date = pd.Timestamp.now().normalize()
        while new_date in log.index:
            new_date += pd.Timedelta(minutes=1)
        log.loc[new_date] = signal_df.iloc[:, -1]
        log.to_csv(filepath)
        print(f"✅ LOGGED: {new_date.date()} | Total: {len(log)}")
        return log


# =============================================================================
# EXECUTE FORCE REBALANCE NOW
# =============================================================================

#🚨 UNCOMMENT TO FORCE REBALANCE (WHEN NEEDED):
trades_df, weights_df, trade_details_df = force_rebalance(1000000)
print("\n📋 COPY THESE TO IBKR PAPER:")
print(trades_df[['action', 'shares']])

def daily_pipeline(portfolio_value=1000000):
    print("🚀 HMM $1M PAPER TRADING - DAILY UPDATE")
    print("=" * 60)
    
    # Check rebalance timing
    filepath = "../data/processed/actual_trades_log.csv"
    if not os.path.exists(filepath):
        print("📄 Run force_rebalance() first")
        return
    
    log = pd.read_csv(filepath, index_col=0, parse_dates=True)
    last_trade = log.index[-1]
    next_rebalance = last_trade + BDay(21)
    days_left = (next_rebalance - pd.Timestamp.now()).days
    
    print(f"📊 Trades logged: {len(log)} days")
    print(f"⏱️  Last: {last_trade.date()} → Next: {next_rebalance.date()} ({days_left} days)")
    
    if days_left <= 0:
        print("🚨 REBALANCE DAY! Run force_rebalance(1000000)")
    else:
        print(f"⏳ HOLDING ({days_left} days)")
    
    # ✅ DAILY PERFORMANCE TRACKING
    print("\n📈 DAILY PERFORMANCE UPDATE:")
    log_daily_performance(portfolio_value)
    show_latest_performance()
    
    print("\n✅ DAILY COMPLETE! Next: 5AM tomorrow")

# RUN DAILY
daily_pipeline(1000000)

Model is not converging.  Current: -505.32002238099864 is not greater than -505.3200201828867. Delta is -2.1981119289193884e-06


🚨 FORCE $1M HMM REBALANCE
🚀 Rolling (drop NaN regimes): T=500, window=252
t=252: 252 raw → 252 valid regimes (100%)
  Regime 0: 227 days ✓
  Regime 1: 25 days ✓


Model is not converging.  Current: -488.49441467638115 is not greater than -488.4944071597059. Delta is -7.516675282204233e-06
Model is not converging.  Current: -470.84717380555395 is not greater than -470.8471719843846. Delta is -1.8211693486591685e-06
Model is not converging.  Current: -431.0582090294358 is not greater than -431.0582083888391. Delta is -6.405966814782005e-07
Model is not converging.  Current: -456.3159603372643 is not greater than -456.31595826196093. Delta is -2.0753033709297597e-06
Model is not converging.  Current: -434.09469870871845 is not greater than -434.09469791651435. Delta is -7.922041049823747e-07


t=273: 252 raw → 252 valid regimes (100%)
  Regime 0: 71 days ✓
  Regime 1: 181 days ✓
t=294: 252 raw → 252 valid regimes (100%)
  Regime 0: 232 days ✓
  Regime 1: only 20 days → equal weights
t=315: 252 raw → 252 valid regimes (100%)
  Regime 0: 228 days ✓
  Regime 1: 24 days ✓


Model is not converging.  Current: -433.5580535964333 is not greater than -433.55805301070717. Delta is -5.857261271557945e-07
Model is not converging.  Current: -464.4662750524257 is not greater than -464.46627293113335. Delta is -2.12129236842884e-06


t=336: 252 raw → 252 valid regimes (100%)
  Regime 0: 228 days ✓
  Regime 1: 24 days ✓
t=357: 252 raw → 252 valid regimes (100%)
  Regime 0: 229 days ✓
  Regime 1: 23 days ✓
t=378: 252 raw → 252 valid regimes (100%)
  Regime 0: 252 days ✓
  Regime 1: only 0 days → equal weights
t=399: 252 raw → 252 valid regimes (100%)
  Regime 0: 228 days ✓


Model is not converging.  Current: -465.498955584707 is not greater than -465.49895342967824. Delta is -2.155028766992473e-06
Model is not converging.  Current: -466.5580035108543 is not greater than -466.5580013871429. Delta is -2.1237113969618804e-06
Model is not converging.  Current: -360.2329030586579 is not greater than -360.23289594703664. Delta is -7.111621243893751e-06
Model is not converging.  Current: -366.3595838406257 is not greater than -366.35958074436235. Delta is -3.096263355928386e-06


  Regime 1: 24 days ✓
t=420: 252 raw → 252 valid regimes (100%)
  Regime 0: 151 days ✓
  Regime 1: 101 days ✓
t=441: 252 raw → 252 valid regimes (100%)
  Regime 0: 66 days ✓
  Regime 1: 186 days ✓
t=462: 252 raw → 252 valid regimes (100%)


Model is not converging.  Current: -384.4426032610578 is not greater than -384.442602664097. Delta is -5.969607741462823e-07
Model is not converging.  Current: -502.142865967413 is not greater than -502.1428645744399. Delta is -1.3929731039752369e-06


  Regime 0: 229 days ✓
  Regime 1: 23 days ✓
t=483: 252 raw → 252 valid regimes (100%)
  Regime 0: 228 days ✓
  Regime 1: 24 days ✓

✅ 12 valid periods generated!
Regime breakdown:
regime
0    10
1     2
Name: count, dtype: int64
📊 HMM WEIGHTS:
AAPL    0.080
AMZN   -0.031
EFA     0.379
META   -0.040
NVDA    0.057
SPY     0.065
TAIL    0.669
USMV   -0.000
VIXY   -0.077
VNQ    -0.101
Name: 2026-02-04 00:00:00, dtype: float64
📖 Loaded 2 rows
💾 Saved 3 rows
📊 Assets: ['AAPL', 'AMZN', 'EFA', 'META', 'NVDA', 'SPY', 'TAIL', 'TSLA', 'USMV', 'VIXY', 'VNQ', 'XLV']
✅ 12 assets | Gross: $1,499,999

🎯 LIMIT + STOP ORDERS (IBKR Paper):
🔍 DELTA CONFIRM:
AAPL: +8% (matches trade_size=$79,664)
AMZN: -1% (matches trade_size=$-12,808)
EFA: +8% (matches trade_size=$81,298)
META: -3% (matches trade_size=$-31,962)
NVDA: +5% (matches trade_size=$48,610)
SPY: -17% (matches trade_size=$-173,623)
TAIL: +4% (matches trade_size=$37,226)
TSLA: +1% (matches trade_size=$12,053)
USMV: -0% (matches trade_size=$-6)
VIX

In [2]:
trade_details

NameError: name 'trade_details' is not defined